# Paper 4, third revision runs

Two parts. Each can run in its own runtime; set the switches in the next cell.

1. **Part 1 (L4).** In this order: (a) the open decision models `laya-ml`, `kev-0.8b`, `decider-2b`, `this-that-1.0`, `kev-9b` and `nimble-9b` on `d2_k150_oos`, the intent benchmark with an explicit out-of-scope option `o151`; (b) the Qwen3-14B generative comparator `comparator-open` on the same condition (switch `RUN_COMPARATOR`); (c) the untuned Qwen3.5 backbones scored on option descriptions, `backbone-desc-qwen35-*`, on `d1_neutral`, `d2_k150` and the four D3 tasks (switch `RUN_BACKBONE_DESC`). `laya-en` is not run: its English head refuses every 150-option question.
2. **Part 2 (A100).** Qwen3.6-27B-FP8 through vLLM, as `comparator-open2` (verbalized JSON) and `comparator-open2-ll` (option-key likelihood), on `d2_k150_oos`.

Before Run all, upload `revision3_upload.zip` to `My Drive/Jev/paper4/colab/`. The notebook unpacks it and checks every file against its SHA-256. Answers are written line by line to `My Drive/Jev/paper4/colab/answers_revision3/`, so a disconnect loses nothing: run all again and finished requests are skipped. A failing condition or model never stops the others. The last cell lists every planned file and writes `answers_revision3.zip` next to the folder for hand-back.

In [ ]:
import subprocess as _sp, sys as _sys
_sp.run([_sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=False)
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, datetime

# ---- switches: part 1 on an L4 runtime, part 2 on an A100 runtime (or both on an A100)
RUN_PART1 = True          # decision models, then the switches below
RUN_COMPARATOR = True     # part 1b: comparator-open (Qwen3-14B-AWQ) on d2_k150_oos
RUN_BACKBONE_DESC = True  # part 1c: the four description-scoring backbones
RUN_DECISION_MODELS = True  # part 1a; False reruns only parts 1b and 1c
DESC_ONLY = None          # or a list, e.g. ['backbone-desc-qwen35-9b-base', 'backbone-desc-qwen35-9b']
RUN_PART2 = True          # Qwen3.6-27B comparator, both modes
COMPARATOR2_VARIANT = 'fp8'   # 'fp8' on an A100; 'awq' is the L4 fallback (tags *-awq*)
# compute units per hour, as Colab shows them in the Resources panel (assumptions until checked)
UNITS_PER_HOUR = {'L4': 4.8, 'A100': 11.8}

LOCAL_ROOT = '/content/p4'
SHARED_LOCAL = f'{LOCAL_ROOT}/shared'
SRC_DIR = f'{SHARED_LOCAL}/src'
INPUTS_DIR = f'{SHARED_LOCAL}/colab/inputs'
os.environ['HF_HOME'] = '/content/hf'
os.environ['USE_TF'] = '0'

DRIVE_ROOT = '/content/drive/MyDrive/Jev/paper4/colab'
UPLOAD_ZIP = f'{DRIVE_ROOT}/revision3_upload.zip'
ANSWERS_DIR = f'{DRIVE_ROOT}/answers_revision3'
os.makedirs(ANSWERS_DIR, exist_ok=True)
# the run log accumulates across the two runtimes
_rl = f'{ANSWERS_DIR}/run_log_revision3.json'
run_log = json.load(open(_rl)) if os.path.exists(_rl) else {}
run_log.setdefault('runtimes', []).append({'start': datetime.datetime.utcnow().isoformat() + 'Z',
                                           'part1': RUN_PART1, 'comparator': RUN_COMPARATOR,
                                           'backbone_desc': RUN_BACKBONE_DESC, 'part2': RUN_PART2})
run_log['units_per_hour_assumed'] = UNITS_PER_HOUR
print('answers on Drive:', ANSWERS_DIR, '| part 1:', RUN_PART1, '(comparator', RUN_COMPARATOR,
      '| backbones', RUN_BACKBONE_DESC, ') | part 2:', RUN_PART2)

In [ ]:
import subprocess, shutil
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                           '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
run_log['runtimes'][-1]['gpu'] = gpu_info
if RUN_PART1: run_log['part1_gpu'] = gpu_info
if RUN_PART2: run_log['part2_gpu'] = gpu_info
print(gpu_info)
print(f"free disk under /content: {shutil.disk_usage('/content').free / 1e9:.0f} GB")
if RUN_PART2 and COMPARATOR2_VARIANT == 'fp8' and 'A100' not in gpu_info and 'H100' not in gpu_info:
    print('WARNING: part 2 with the FP8 checkpoint needs 40 GB (an A100). On an L4 either '
          'switch to an A100 runtime or set COMPARATOR2_VARIANT = "awq" (tags *-awq*).')
if RUN_PART1 and not RUN_PART2 and 'L4' not in gpu_info:
    print('note: part 1 was planned for an L4; latencies will not match the earlier runs.')

## Unpack the upload and verify every file

In [ ]:
import hashlib, zipfile, shutil

EXPECTED = {
 "d1_neutral": {
  "file": "d1_neutral.jsonl",
  "requests": 400,
  "sha256": "013fd3993a5d3e4e77cd267161d6333a202a35ca0daa2342abaf945f8548b591"
 },
 "d2_k150": {
  "file": "d2_k150.jsonl",
  "requests": 800,
  "sha256": "b41706d09f2bd228463d50b5e1cd81d45c7da2fd3abaa5ce7fd869aaabd9a1ec"
 },
 "d2_k150_oos": {
  "file": "d2_k150_oos.jsonl",
  "requests": 800,
  "sha256": "7d991da0c4f71063e92ecac3c21cf2d7bcc5ee5ffbecdae40a5797e9b59ba73a"
 },
 "d3_conv_go_awry": {
  "file": "d3_conv_go_awry.jsonl",
  "requests": 500,
  "sha256": "de3ef7269832338cee4c271d23b9a4a18ea9919d55937bb1eb26612ee29fe712"
 },
 "d3_emotion": {
  "file": "d3_emotion.jsonl",
  "requests": 498,
  "sha256": "ccd5a620b9269d577a0f9672351b0eaa1b2f03fd0527eadaf9884e3078cc9f9b"
 },
 "d3_wiki_corpus": {
  "file": "d3_wiki_corpus.jsonl",
  "requests": 500,
  "sha256": "8b24c2f580b834d61971029eb4b8af94fc58d186b4673f3fe2a57b33557436d3"
 },
 "d3_wiki_politeness": {
  "file": "d3_wiki_politeness.jsonl",
  "requests": 498,
  "sha256": "a98b22a9efbac95da3dca46eaa40f5ac0b3f024527aa823f8e73446f74596e68"
 }
}
SRC_SHA256 = {
 "adapters.py": "360d1110ba505ee68a52ff2b247f82cdb867c314edcb730c867537df5b57dfc0",
 "common.py": "5fc5d29c76ac4b1459161ad74b199c865fbfe5cf0d49767ab7b5b985e0a3aaca",
 "harness.py": "2be4b3216600dc518126c62b1299cb6abe3e6626fb496fc8e4862d1f276af162"
}
ZIP_SHA256 = '79a05cc4c83090066111d31719c9e438ef8010660ece69a4af6560a6eda376fa'

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if not os.path.exists(UPLOAD_ZIP):
    raise SystemExit(f'{UPLOAD_ZIP} not found: upload revision3_upload.zip to that Drive folder first.')
got = _sha256(UPLOAD_ZIP)
if got != ZIP_SHA256:
    print(f'note: the zip differs from the one this notebook was built with ({got}); '
          'the per-file check below decides')
shutil.rmtree(LOCAL_ROOT, ignore_errors=True)
with zipfile.ZipFile(UPLOAD_ZIP) as z:
    z.extractall(LOCAL_ROOT)
bad = []
for cond, meta in EXPECTED.items():
    p = f'{INPUTS_DIR}/' + meta['file']
    if not os.path.exists(p) or _sha256(p) != meta['sha256']:
        bad.append(cond)
for f, h in SRC_SHA256.items():
    if _sha256(f'{SRC_DIR}/{f}') != h:
        bad.append(f)
if bad:
    raise SystemExit(f'these files do not match the build: {bad} -- stopping.')
print(f'{len(EXPECTED)} input files and {len(SRC_SHA256)} source files verified.')

## Revisions, plan and helpers

Every checkpoint is pinned through `P4_REVISIONS`, which `adapters.py` reads: the decision models and Qwen3-14B-AWQ to the commits of their original answers (as in the first revision run), the backbones and Qwen3.6-27B to the commits of the second revision run. The description-scoring tags load the same backbone checkpoints as the key-scoring tags.

In [ ]:
REVISIONS = {
 "Mapika/decider-2b": "d61c1c16089572df5d180329b9fea4997a90090c",
 "Qwen/Qwen3-14B-AWQ": "31c69efc29464b6bb0aee1398b5a7b50a99340c3",
 "Qwen/Qwen3.5-0.8B-Base": "dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68",
 "Qwen/Qwen3.5-2B-Base": "b1485b2fa6dfa1287294f269f5fb618e03d52d7c",
 "Qwen/Qwen3.5-9B": "c202236235762e1c871ad0ccb60c8ee5ba337b9a",
 "Qwen/Qwen3.5-9B-Base": "68c46c4b3498877f3ef123c856ecfde50c39f404",
 "Qwen/Qwen3.6-27B-FP8": "e89b16ebf1988b3d6befa7de50abc2d76f26eb09",
 "bespokelabs/Bespoke-Nimble-9B": "bd792f44ec8e265be861bfcdf4e05967ffe0e858",
 "convaiinnovations/laya": "55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851",
 "cyankiwi/Qwen3.6-27B-AWQ-INT4": "e5cc0400fb2403c437c2c40a7c52fb5ae93fda18",
 "flock-io/this-that-model-1.0": "3d927195c4f9845efe66c5715883a7a0f42b1239",
 "jaredpalmer/kev-0.8b": "9a45d25eb2ab761841196625383fa1dff0e56c1e",
 "jaredpalmer/kev-9b": "2629c06a5aeb0feb3b9783bafed17ed8f39ecf5c"
}
TAG_REPO = {
 "laya-ml": "convaiinnovations/laya",
 "kev-0.8b": "jaredpalmer/kev-0.8b",
 "decider-2b": "Mapika/decider-2b",
 "this-that-1.0": "flock-io/this-that-model-1.0",
 "kev-9b": "jaredpalmer/kev-9b",
 "nimble-9b": "bespokelabs/Bespoke-Nimble-9B",
 "comparator-open": "Qwen/Qwen3-14B-AWQ",
 "backbone-desc-qwen35-0.8b-base": "Qwen/Qwen3.5-0.8B-Base",
 "backbone-desc-qwen35-2b-base": "Qwen/Qwen3.5-2B-Base",
 "backbone-desc-qwen35-9b-base": "Qwen/Qwen3.5-9B-Base",
 "backbone-desc-qwen35-9b": "Qwen/Qwen3.5-9B",
 "comparator-open2": "Qwen/Qwen3.6-27B-FP8",
 "comparator-open2-ll": "Qwen/Qwen3.6-27B-FP8",
 "comparator-open2-awq": "cyankiwi/Qwen3.6-27B-AWQ-INT4",
 "comparator-open2-awq-ll": "cyankiwi/Qwen3.6-27B-AWQ-INT4"
}
OOS = 'd2_k150_oos'
DECISION_TAGS = ['laya-ml', 'kev-0.8b', 'decider-2b', 'this-that-1.0', 'kev-9b', 'nimble-9b']
DESC_TAGS = ['backbone-desc-qwen35-0.8b-base', 'backbone-desc-qwen35-2b-base', 'backbone-desc-qwen35-9b-base', 'backbone-desc-qwen35-9b']
DESC_CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness']
PART2_CONDS = ['d2_k150_oos']
COMPARATOR2_TAGS = {
 "fp8": {
  "verbal": "comparator-open2",
  "ll": "comparator-open2-ll"
 },
 "awq": {
  "verbal": "comparator-open2-awq",
  "ll": "comparator-open2-awq-ll"
 }
}

def answer_counts(path):
    """(answered lines, error lines) of one answer file; (0, 0) when it does not exist."""
    ok = err = 0
    if os.path.exists(path):
        with open(path, encoding='utf-8') as fh:
            for l in fh:
                try:
                    j = json.loads(l)
                except json.JSONDecodeError:
                    continue
                if j.get('error'):
                    err += 1
                else:
                    ok += 1
    return ok, err

def smoke_report(smoke_dir):
    """Error lines and the largest self-check difference over every smoke answer file.
    harness.py records a failed request, a failed self-check included, as an error line and
    exits 0, so the smoke gate reads the files instead of trusting the exit code."""
    errs, dmax = [], None
    for root, _dirs, files in os.walk(smoke_dir):
        for fn in sorted(files):
            if not fn.endswith('.jsonl'):
                continue
            with open(os.path.join(root, fn), encoding='utf-8') as fh:
                for l in fh:
                    try:
                        j = json.loads(l)
                    except json.JSONDecodeError:
                        continue
                    if j.get('error'):
                        errs.append(f"{fn}: {j['error'][:200]}")
                        continue
                    for a in (j.get('answers') or {}).values():
                        d = (a.get('raw') or {}).get('check_max_abs_dprob')
                        if d is not None:
                            dmax = d if dmax is None else max(dmax, d)
    return errs, dmax

print(json.dumps(REVISIONS, indent=1))

## Part 1a: open decision models on d2_k150_oos

Each cell builds (or reuses) its uv env, smoke-tests 5 requests (any error line stops that model), prints a time estimate, then runs the 800 requests. Output is streamed and appended to `answers_revision3/<tag>/run_log_<tag>.txt`. Look for `DONE_revision3` or `ERROR_revision3.txt` in each model folder.

### laya-ml

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'laya-ml'
ENV_NAME = 'laya'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/laya --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/laya/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 laya==0.3.20 python-dotenv httpx numpy\n'
JOBS = [('laya-ml', 'd2_k150_oos', 1)]   # [(run tag, condition, rep), ...]

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, log_path, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

if not (RUN_PART1 and globals().get('RUN_DECISION_MODELS', True)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_DECISION_MODELS = '
          f'{globals().get("RUN_DECISION_MODELS", True)})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    os.makedirs(SMOKE_DIR, exist_ok=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
        rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, LOG, limit=5, answers_dir=SMOKE_DIR)
        if rc:
            raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
        errs, _ = smoke_report(SMOKE_DIR)
        if errs:
            raise RuntimeError(f'{TAG}: {len(errs)} smoke request(s) returned an error line, '
                               f'first: {errs[0]}')
        n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
        rate = UNITS_PER_HOUR['L4']
        print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
              f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
              f'~{dt / 5 * n_req / 3600 * rate:.1f} compute units at {rate}/h on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': dt / 5,
                                                 'requests': n_req}

        failed = []
        for run_tag, cond, rep in JOBS:
            rc, dt = run_harness(run_tag, cond, rep, LOG)
            ok, err = answer_counts(f'{ANSWERS_DIR}/{run_tag}/{cond}__rep{rep}.jsonl')
            want = EXPECTED[cond]['requests']
            print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min, {ok}/{want} '
                  f'answered, {err} error lines', flush=True)
            run_log['models'][TAG][f'{run_tag}/{cond}_min'] = dt / 60
            if rc or ok < want:
                failed.append(f'{run_tag}/{cond}__rep{rep}')
        if failed:
            raise RuntimeError(f'{TAG}: conditions incomplete: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### kev-0.8b

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'kev-0.8b'
ENV_NAME = 'kev_decider_tt'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('kev-0.8b', 'd2_k150_oos', 1)]   # [(run tag, condition, rep), ...]

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, log_path, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

if not (RUN_PART1 and globals().get('RUN_DECISION_MODELS', True)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_DECISION_MODELS = '
          f'{globals().get("RUN_DECISION_MODELS", True)})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    os.makedirs(SMOKE_DIR, exist_ok=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
        rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, LOG, limit=5, answers_dir=SMOKE_DIR)
        if rc:
            raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
        errs, _ = smoke_report(SMOKE_DIR)
        if errs:
            raise RuntimeError(f'{TAG}: {len(errs)} smoke request(s) returned an error line, '
                               f'first: {errs[0]}')
        n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
        rate = UNITS_PER_HOUR['L4']
        print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
              f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
              f'~{dt / 5 * n_req / 3600 * rate:.1f} compute units at {rate}/h on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': dt / 5,
                                                 'requests': n_req}

        failed = []
        for run_tag, cond, rep in JOBS:
            rc, dt = run_harness(run_tag, cond, rep, LOG)
            ok, err = answer_counts(f'{ANSWERS_DIR}/{run_tag}/{cond}__rep{rep}.jsonl')
            want = EXPECTED[cond]['requests']
            print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min, {ok}/{want} '
                  f'answered, {err} error lines', flush=True)
            run_log['models'][TAG][f'{run_tag}/{cond}_min'] = dt / 60
            if rc or ok < want:
                failed.append(f'{run_tag}/{cond}__rep{rep}')
        if failed:
            raise RuntimeError(f'{TAG}: conditions incomplete: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### decider-2b

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'decider-2b'
ENV_NAME = 'kev_decider_tt'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('decider-2b', 'd2_k150_oos', 1)]   # [(run tag, condition, rep), ...]

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, log_path, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

if not (RUN_PART1 and globals().get('RUN_DECISION_MODELS', True)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_DECISION_MODELS = '
          f'{globals().get("RUN_DECISION_MODELS", True)})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    os.makedirs(SMOKE_DIR, exist_ok=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
        rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, LOG, limit=5, answers_dir=SMOKE_DIR)
        if rc:
            raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
        errs, _ = smoke_report(SMOKE_DIR)
        if errs:
            raise RuntimeError(f'{TAG}: {len(errs)} smoke request(s) returned an error line, '
                               f'first: {errs[0]}')
        n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
        rate = UNITS_PER_HOUR['L4']
        print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
              f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
              f'~{dt / 5 * n_req / 3600 * rate:.1f} compute units at {rate}/h on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': dt / 5,
                                                 'requests': n_req}

        failed = []
        for run_tag, cond, rep in JOBS:
            rc, dt = run_harness(run_tag, cond, rep, LOG)
            ok, err = answer_counts(f'{ANSWERS_DIR}/{run_tag}/{cond}__rep{rep}.jsonl')
            want = EXPECTED[cond]['requests']
            print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min, {ok}/{want} '
                  f'answered, {err} error lines', flush=True)
            run_log['models'][TAG][f'{run_tag}/{cond}_min'] = dt / 60
            if rc or ok < want:
                failed.append(f'{run_tag}/{cond}__rep{rep}')
        if failed:
            raise RuntimeError(f'{TAG}: conditions incomplete: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### this-that-1.0

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'this-that-1.0'
ENV_NAME = 'kev_decider_tt'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f' 'thisthat @ git+https://github.com/FLock-io/this-that-model@542d445efa5f68b14bfbd1f8ed25aacd8379d839'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('this-that-1.0', 'd2_k150_oos', 1)]   # [(run tag, condition, rep), ...]

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, log_path, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

if not (RUN_PART1 and globals().get('RUN_DECISION_MODELS', True)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_DECISION_MODELS = '
          f'{globals().get("RUN_DECISION_MODELS", True)})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    os.makedirs(SMOKE_DIR, exist_ok=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
        rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, LOG, limit=5, answers_dir=SMOKE_DIR)
        if rc:
            raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
        errs, _ = smoke_report(SMOKE_DIR)
        if errs:
            raise RuntimeError(f'{TAG}: {len(errs)} smoke request(s) returned an error line, '
                               f'first: {errs[0]}')
        n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
        rate = UNITS_PER_HOUR['L4']
        print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
              f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
              f'~{dt / 5 * n_req / 3600 * rate:.1f} compute units at {rate}/h on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': dt / 5,
                                                 'requests': n_req}

        failed = []
        for run_tag, cond, rep in JOBS:
            rc, dt = run_harness(run_tag, cond, rep, LOG)
            ok, err = answer_counts(f'{ANSWERS_DIR}/{run_tag}/{cond}__rep{rep}.jsonl')
            want = EXPECTED[cond]['requests']
            print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min, {ok}/{want} '
                  f'answered, {err} error lines', flush=True)
            run_log['models'][TAG][f'{run_tag}/{cond}_min'] = dt / 60
            if rc or ok < want:
                failed.append(f'{run_tag}/{cond}__rep{rep}')
        if failed:
            raise RuntimeError(f'{TAG}: conditions incomplete: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### kev-9b

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'kev-9b'
ENV_NAME = 'kev_decider_tt'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('kev-9b', 'd2_k150_oos', 1)]   # [(run tag, condition, rep), ...]

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, log_path, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

if not (RUN_PART1 and globals().get('RUN_DECISION_MODELS', True)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_DECISION_MODELS = '
          f'{globals().get("RUN_DECISION_MODELS", True)})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    os.makedirs(SMOKE_DIR, exist_ok=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
        rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, LOG, limit=5, answers_dir=SMOKE_DIR)
        if rc:
            raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
        errs, _ = smoke_report(SMOKE_DIR)
        if errs:
            raise RuntimeError(f'{TAG}: {len(errs)} smoke request(s) returned an error line, '
                               f'first: {errs[0]}')
        n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
        rate = UNITS_PER_HOUR['L4']
        print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
              f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
              f'~{dt / 5 * n_req / 3600 * rate:.1f} compute units at {rate}/h on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': dt / 5,
                                                 'requests': n_req}

        failed = []
        for run_tag, cond, rep in JOBS:
            rc, dt = run_harness(run_tag, cond, rep, LOG)
            ok, err = answer_counts(f'{ANSWERS_DIR}/{run_tag}/{cond}__rep{rep}.jsonl')
            want = EXPECTED[cond]['requests']
            print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min, {ok}/{want} '
                  f'answered, {err} error lines', flush=True)
            run_log['models'][TAG][f'{run_tag}/{cond}_min'] = dt / 60
            if rc or ok < want:
                failed.append(f'{run_tag}/{cond}__rep{rep}')
        if failed:
            raise RuntimeError(f'{TAG}: conditions incomplete: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### nimble-9b

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'nimble-9b'
ENV_NAME = 'nimble'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/nimble --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/nimble/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 peft==0.21.0 accelerate==1.15.0 sentencepiece==0.2.2 pillow==12.3.0 'huggingface_hub>=0.34' python-dotenv httpx numpy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/nimble/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('nimble-9b', 'd2_k150_oos', 1)]   # [(run tag, condition, rep), ...]

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, log_path, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

if not (RUN_PART1 and globals().get('RUN_DECISION_MODELS', True)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_DECISION_MODELS = '
          f'{globals().get("RUN_DECISION_MODELS", True)})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    os.makedirs(SMOKE_DIR, exist_ok=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
        rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, LOG, limit=5, answers_dir=SMOKE_DIR)
        if rc:
            raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
        errs, _ = smoke_report(SMOKE_DIR)
        if errs:
            raise RuntimeError(f'{TAG}: {len(errs)} smoke request(s) returned an error line, '
                               f'first: {errs[0]}')
        n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
        rate = UNITS_PER_HOUR['L4']
        print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
              f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
              f'~{dt / 5 * n_req / 3600 * rate:.1f} compute units at {rate}/h on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': dt / 5,
                                                 'requests': n_req}

        failed = []
        for run_tag, cond, rep in JOBS:
            rc, dt = run_harness(run_tag, cond, rep, LOG)
            ok, err = answer_counts(f'{ANSWERS_DIR}/{run_tag}/{cond}__rep{rep}.jsonl')
            want = EXPECTED[cond]['requests']
            print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min, {ok}/{want} '
                  f'answered, {err} error lines', flush=True)
            run_log['models'][TAG][f'{run_tag}/{cond}_min'] = dt / 60
            if rc or ok < want:
                failed.append(f'{run_tag}/{cond}__rep{rep}')
        if failed:
            raise RuntimeError(f'{TAG}: conditions incomplete: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


## Part 1b: generative comparator on d2_k150_oos

Qwen3-14B-AWQ through vLLM, tag `comparator-open`, with the settings of its earlier runs (greedy, thinking off, verbalized JSON). vLLM is pinned to 0.30.0, the version its first run logged. Switch `RUN_COMPARATOR` in the first code cell.

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'comparator-open'
ENV_NAME = 'vllm_env'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
CONDS = ['d2_k150_oos']

def stream(cmd, log_path, env=None):
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        return p.wait()

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    env['VLLM_LOGGING_LEVEL'] = 'INFO'
    # FlashInfer JIT-compiles kernels with ninja, which lives in the env's bin folder;
    # greedy decoding does not need the FlashInfer sampler, so it is switched off as well
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    env['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
    return env

if not (RUN_PART1 and RUN_COMPARATOR):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_COMPARATOR = {RUN_COMPARATOR})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL + f'{PYBIN} -c "import vllm, torch; print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"'], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc})')
        print(f'=== {TAG}: smoke test, 5 requests of {CONDS[0]} ===', flush=True)
        t0 = time.time()
        rc = stream([PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', CONDS[0],
                     '--rep', '1', '--limit', '5'], LOG, env=harness_env(SMOKE_DIR))
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_load': time.time() - t0}
        if rc:
            raise RuntimeError(f'smoke test failed (exit {rc}); see {LOG}')
        errs, _ = smoke_report(SMOKE_DIR)
        if errs:
            raise RuntimeError(f'{len(errs)} smoke request(s) returned an error line, first: {errs[0]}')
        print(f'=== {TAG}: full run (one process, model loaded once) ===', flush=True)
        t0 = time.time()
        rc = stream([PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', ','.join(CONDS),
                     '--rep', '1'], LOG, env=harness_env(ANSWERS_DIR))
        run_log['models'][TAG]['full_min'] = (time.time() - t0) / 60
        print(f'full run exit {rc}, {(time.time() - t0) / 60:.1f} min', flush=True)
        missing = [c for c in CONDS
                   if answer_counts(f'{TAG_DIR}/{c}__rep1.jsonl')[0] < EXPECTED[c]['requests']]
        if rc or missing:
            raise RuntimeError(f'full run exit {rc}, incomplete: {missing}; see {LOG}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing; the full log is in {LOG} ===')
        print(tb)
    finally:
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


## Part 1c: backbones scored on option descriptions

Each cell builds (or reuses) the `backbone` uv env, runs a smoke test of 2 requests on each of `d2_k150`, `d1_neutral` and `d3_emotion` with the self-check on (every description is rescored by a plain full-sequence forward pass; the two option distributions must agree within 0.15), then runs all six conditions in one process. A condition left incomplete is retried on its own. Switch `RUN_BACKBONE_DESC` in the first code cell.

### backbone-desc-qwen35-0.8b-base

Qwen/Qwen3.5-0.8B-Base, the base of kev-0.8b.

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'backbone-desc-qwen35-0.8b-base'
ENV_NAME = 'backbone'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness']   # rep 1 each
SMOKE_CONDS = ['d2_k150', 'd1_neutral', 'd3_emotion']

def harness_env(answers_dir, check=0):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    env['P4_OPTSCORE_CHECK'] = str(check)
    # rows per scoring pass; the 9B checkpoints leave little room on an L4 (adapters retries
    # with fewer rows on out-of-memory in any case)
    env['P4_DESC_ROWS'] = '4' if '9b' in TAG else '16'
    # ninja and any other console script of the env must be on PATH (Triton, fla kernels)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(conds, log_path, limit=None, answers_dir=None, check=0):
    """Conditions in one process (the model loads once); output goes to the cell and the log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', ','.join(conds), '--rep', '1']
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR, check),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

def n_ok(cond):
    return answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')[0]

if not (RUN_PART1 and RUN_BACKBONE_DESC
        and (globals().get('DESC_ONLY') is None or TAG in DESC_ONLY)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_BACKBONE_DESC = {RUN_BACKBONE_DESC}, '
          f'DESC_ONLY = {globals().get("DESC_ONLY")})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    # a rerun replaces the marker of an earlier failed attempt of this same notebook
    if os.path.exists(f'{TAG_DIR}/ERROR_revision3.txt'):
        os.rename(f'{TAG_DIR}/ERROR_revision3.txt', f'{TAG_DIR}/ERROR_revision3_earlier.txt')
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test with the cached-versus-full-sequence self-check '
              f'(2 requests each of {SMOKE_CONDS}) ===', flush=True)
        rc, dt = run_harness(SMOKE_CONDS, LOG, limit=2, answers_dir=SMOKE_DIR, check=100)
        errs, dmax = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt:.0f} s for {2 * len(SMOKE_CONDS)} requests including the model '
              f'load; largest self-check difference {dmax}', flush=True)
        if rc or errs:
            raise RuntimeError(f'{TAG}: smoke test or self-check failed (exit {rc}, '
                               f'{len(errs)} error lines, first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'{TAG}: the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_load': dt, 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}

        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = run_harness(CONDS, LOG)
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        # a condition left incomplete (a crash, a disconnect) is retried on its own, so one
        # failing condition never blocks the others; finished requests are skipped
        failed = []
        for c in CONDS:
            if n_ok(c) < EXPECTED[c]['requests']:
                rc, dt = run_harness([c], LOG)
                print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
                if n_ok(c) < EXPECTED[c]['requests']:
                    failed.append(c)
        if failed:
            raise RuntimeError(f'{TAG}: incomplete conditions: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### backbone-desc-qwen35-2b-base

Qwen/Qwen3.5-2B-Base, the base of decider-2b and this-that-1.0.

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'backbone-desc-qwen35-2b-base'
ENV_NAME = 'backbone'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness']   # rep 1 each
SMOKE_CONDS = ['d2_k150', 'd1_neutral', 'd3_emotion']

def harness_env(answers_dir, check=0):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    env['P4_OPTSCORE_CHECK'] = str(check)
    # rows per scoring pass; the 9B checkpoints leave little room on an L4 (adapters retries
    # with fewer rows on out-of-memory in any case)
    env['P4_DESC_ROWS'] = '4' if '9b' in TAG else '16'
    # ninja and any other console script of the env must be on PATH (Triton, fla kernels)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(conds, log_path, limit=None, answers_dir=None, check=0):
    """Conditions in one process (the model loads once); output goes to the cell and the log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', ','.join(conds), '--rep', '1']
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR, check),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

def n_ok(cond):
    return answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')[0]

if not (RUN_PART1 and RUN_BACKBONE_DESC
        and (globals().get('DESC_ONLY') is None or TAG in DESC_ONLY)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_BACKBONE_DESC = {RUN_BACKBONE_DESC}, '
          f'DESC_ONLY = {globals().get("DESC_ONLY")})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    # a rerun replaces the marker of an earlier failed attempt of this same notebook
    if os.path.exists(f'{TAG_DIR}/ERROR_revision3.txt'):
        os.rename(f'{TAG_DIR}/ERROR_revision3.txt', f'{TAG_DIR}/ERROR_revision3_earlier.txt')
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test with the cached-versus-full-sequence self-check '
              f'(2 requests each of {SMOKE_CONDS}) ===', flush=True)
        rc, dt = run_harness(SMOKE_CONDS, LOG, limit=2, answers_dir=SMOKE_DIR, check=100)
        errs, dmax = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt:.0f} s for {2 * len(SMOKE_CONDS)} requests including the model '
              f'load; largest self-check difference {dmax}', flush=True)
        if rc or errs:
            raise RuntimeError(f'{TAG}: smoke test or self-check failed (exit {rc}, '
                               f'{len(errs)} error lines, first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'{TAG}: the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_load': dt, 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}

        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = run_harness(CONDS, LOG)
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        # a condition left incomplete (a crash, a disconnect) is retried on its own, so one
        # failing condition never blocks the others; finished requests are skipped
        failed = []
        for c in CONDS:
            if n_ok(c) < EXPECTED[c]['requests']:
                rc, dt = run_harness([c], LOG)
                print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
                if n_ok(c) < EXPECTED[c]['requests']:
                    failed.append(c)
        if failed:
            raise RuntimeError(f'{TAG}: incomplete conditions: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### backbone-desc-qwen35-9b-base

Qwen/Qwen3.5-9B-Base, the base of kev-9b.

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'backbone-desc-qwen35-9b-base'
ENV_NAME = 'backbone'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness']   # rep 1 each
SMOKE_CONDS = ['d2_k150', 'd1_neutral', 'd3_emotion']

def harness_env(answers_dir, check=0):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    env['P4_OPTSCORE_CHECK'] = str(check)
    # rows per scoring pass; the 9B checkpoints leave little room on an L4 (adapters retries
    # with fewer rows on out-of-memory in any case)
    env['P4_DESC_ROWS'] = '4' if '9b' in TAG else '16'
    # ninja and any other console script of the env must be on PATH (Triton, fla kernels)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(conds, log_path, limit=None, answers_dir=None, check=0):
    """Conditions in one process (the model loads once); output goes to the cell and the log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', ','.join(conds), '--rep', '1']
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR, check),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

def n_ok(cond):
    return answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')[0]

if not (RUN_PART1 and RUN_BACKBONE_DESC
        and (globals().get('DESC_ONLY') is None or TAG in DESC_ONLY)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_BACKBONE_DESC = {RUN_BACKBONE_DESC}, '
          f'DESC_ONLY = {globals().get("DESC_ONLY")})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    # a rerun replaces the marker of an earlier failed attempt of this same notebook
    if os.path.exists(f'{TAG_DIR}/ERROR_revision3.txt'):
        os.rename(f'{TAG_DIR}/ERROR_revision3.txt', f'{TAG_DIR}/ERROR_revision3_earlier.txt')
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test with the cached-versus-full-sequence self-check '
              f'(2 requests each of {SMOKE_CONDS}) ===', flush=True)
        rc, dt = run_harness(SMOKE_CONDS, LOG, limit=2, answers_dir=SMOKE_DIR, check=100)
        errs, dmax = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt:.0f} s for {2 * len(SMOKE_CONDS)} requests including the model '
              f'load; largest self-check difference {dmax}', flush=True)
        if rc or errs:
            raise RuntimeError(f'{TAG}: smoke test or self-check failed (exit {rc}, '
                               f'{len(errs)} error lines, first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'{TAG}: the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_load': dt, 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}

        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = run_harness(CONDS, LOG)
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        # a condition left incomplete (a crash, a disconnect) is retried on its own, so one
        # failing condition never blocks the others; finished requests are skipped
        failed = []
        for c in CONDS:
            if n_ok(c) < EXPECTED[c]['requests']:
                rc, dt = run_harness([c], LOG)
                print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
                if n_ok(c) < EXPECTED[c]['requests']:
                    failed.append(c)
        if failed:
            raise RuntimeError(f'{TAG}: incomplete conditions: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### backbone-desc-qwen35-9b

Qwen/Qwen3.5-9B (post-trained), the base of nimble-9b.

In [ ]:
import subprocess, time, json, os, traceback, shutil

TAG = 'backbone-desc-qwen35-9b'
ENV_NAME = 'backbone'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness']   # rep 1 each
SMOKE_CONDS = ['d2_k150', 'd1_neutral', 'd3_emotion']

def harness_env(answers_dir, check=0):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    env['P4_OPTSCORE_CHECK'] = str(check)
    # rows per scoring pass; the 9B checkpoints leave little room on an L4 (adapters retries
    # with fewer rows on out-of-memory in any case)
    env['P4_DESC_ROWS'] = '4' if '9b' in TAG else '16'
    # ninja and any other console script of the env must be on PATH (Triton, fla kernels)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(conds, log_path, limit=None, answers_dir=None, check=0):
    """Conditions in one process (the model loads once); output goes to the cell and the log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', ','.join(conds), '--rep', '1']
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR, check),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

def n_ok(cond):
    return answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')[0]

if not (RUN_PART1 and RUN_BACKBONE_DESC
        and (globals().get('DESC_ONLY') is None or TAG in DESC_ONLY)):
    print(f'{TAG}: skipped (RUN_PART1 = {RUN_PART1}, RUN_BACKBONE_DESC = {RUN_BACKBONE_DESC}, '
          f'DESC_ONLY = {globals().get("DESC_ONLY")})')
else:
    TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
    os.makedirs(TAG_DIR, exist_ok=True)
    # a rerun replaces the marker of an earlier failed attempt of this same notebook
    if os.path.exists(f'{TAG_DIR}/ERROR_revision3.txt'):
        os.rename(f'{TAG_DIR}/ERROR_revision3.txt', f'{TAG_DIR}/ERROR_revision3_earlier.txt')
    LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        subprocess.run(['bash', '-c', INSTALL], check=True)

        print(f'=== {TAG}: smoke test with the cached-versus-full-sequence self-check '
              f'(2 requests each of {SMOKE_CONDS}) ===', flush=True)
        rc, dt = run_harness(SMOKE_CONDS, LOG, limit=2, answers_dir=SMOKE_DIR, check=100)
        errs, dmax = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt:.0f} s for {2 * len(SMOKE_CONDS)} requests including the model '
              f'load; largest self-check difference {dmax}', flush=True)
        if rc or errs:
            raise RuntimeError(f'{TAG}: smoke test or self-check failed (exit {rc}, '
                               f'{len(errs)} error lines, first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'{TAG}: the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_load': dt, 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}

        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = run_harness(CONDS, LOG)
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        # a condition left incomplete (a crash, a disconnect) is retried on its own, so one
        # failing condition never blocks the others; finished requests are skipped
        failed = []
        for c in CONDS:
            if n_ok(c) < EXPECTED[c]['requests']:
                rc, dt = run_harness([c], LOG)
                print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
                if n_ok(c) < EXPECTED[c]['requests']:
                    failed.append(c)
        if failed:
            raise RuntimeError(f'{TAG}: incomplete conditions: {failed}')
        with open(f'{TAG_DIR}/DONE_revision3', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print(f'=== {TAG}: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision3.txt', 'w') as fh:
            fh.write(tb)
        print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision3.txt) ===')
        print(tb)
    finally:
        import gc
        gc.collect()
        subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


## Part 2: stronger comparator on d2_k150_oos

Builds the `vllm_env` env (vLLM 0.30.0), smoke-tests both modes on 5 requests of `d2_k150_oos`, then runs the verbalized mode and the likelihood mode, each in a fresh process.

In [ ]:
import subprocess, time, json, os, traceback, shutil

ENV_NAME = 'vllm_env'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
CONDS = ['d2_k150_oos']
SMOKE_COND = CONDS[0]      # both smoke tests run on the new condition, so o151 is exercised
TAGS = COMPARATOR2_TAGS[COMPARATOR2_VARIANT]

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    env['VLLM_LOGGING_LEVEL'] = 'INFO'
    # FlashInfer JIT-compiles kernels with ninja, which lives in the env's bin folder;
    # greedy decoding does not need the FlashInfer sampler, so it is switched off as well
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    env['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
    return env

def stream(cmd, log_path, env=None):
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        return p.wait()

def harness(tag, conds, limit=None, answers_dir=None):
    """One process per call: the engine starts fresh for each mode."""
    os.makedirs(f'{ANSWERS_DIR}/{tag}', exist_ok=True)
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', tag, '--cond', ','.join(conds), '--rep', '1']
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    rc = stream(cmd, f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt', env=harness_env(answers_dir or ANSWERS_DIR))
    return rc, time.time() - t0

def mark(tag, ok, detail=''):
    name = 'DONE_revision3' if ok else 'ERROR_revision3.txt'
    with open(f'{ANSWERS_DIR}/{tag}/{name}', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n' + detail)

if not RUN_PART2:
    print('part 2 skipped (RUN_PART2 = False)')
else:
    os.makedirs(f'{ANSWERS_DIR}/{TAGS["verbal"]}', exist_ok=True)
    env_log = f'{ANSWERS_DIR}/{TAGS["verbal"]}/run_log_{TAGS["verbal"]}.txt'
    rc = stream(['bash', '-c', INSTALL + f'{PYBIN} -c "import vllm, torch; print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"'], env_log)
    if rc:
        print(f'=== vLLM env build failed (exit {rc}); part 2 stops here, see {env_log} ===')
    else:
        run_log.setdefault('part2', {}).update({'variant': COMPARATOR2_VARIANT, 'tags': TAGS})
        n_q = sum(EXPECTED[c]['requests'] for c in CONDS)
        # smoke tests first: 5 requests of each mode on the condition of the full run
        smoke_ok = True
        for mode in ('verbal', 'll'):
            tag = TAGS[mode]
            sd = f'/content/smoke/{tag}'
            shutil.rmtree(sd, ignore_errors=True)
            print(f'=== {tag}: smoke test, 5 requests of {SMOKE_COND} ===', flush=True)
            rc, dt = harness(tag, [SMOKE_COND], limit=5, answers_dir=sd)
            errs, _ = smoke_report(sd)
            print(f'{tag}: smoke {dt:.0f} s including the engine start (exit {rc}, '
                  f'{len(errs)} error lines)', flush=True)
            run_log['part2'][f'smoke_{mode}_s_incl_load'] = dt
            if rc or errs:
                smoke_ok = False
                mark(tag, False, f'smoke test failed (exit {rc}); {errs[:1]}')
                continue
            # engine time per question from the smoke answers (batch share, no engine start)
            per_q = []
            with open(f'{sd}/{tag}/{SMOKE_COND}__rep1.jsonl', encoding='utf-8') as fh:
                for l in fh:
                    j = json.loads(l)
                    if not j.get('error'):
                        for a in j['answers'].values():
                            r = a.get('raw') or {}
                            if r.get('batch_wall_s') is not None and r.get('n_requests'):
                                per_q.append(r['batch_wall_s'] / r['n_requests'])
            if per_q:
                s = sum(per_q) / len(per_q)
                print(f'{tag}: {s:.2f} s per question in the smoke batch -> about {s * n_q / 60:.0f} min '
                      f'for the {n_q} questions of {CONDS}; small smoke batches overstate batched time',
                      flush=True)
                run_log['part2'][f'smoke_{mode}_s_per_question'] = s
        if not smoke_ok:
            print('=== a comparator smoke test failed; see the run_log_*.txt files. If the log shows '
                  'out-of-memory on an L4, part 2 needs an A100 (or COMPARATOR2_VARIANT = "awq"). ===')
        else:
            for mode in ('verbal', 'll'):
                tag = TAGS[mode]
                print(f'=== {tag}: {CONDS} ===', flush=True)
                try:
                    rc, dt = harness(tag, CONDS)
                    missing = [c for c in CONDS
                               if answer_counts(f'{ANSWERS_DIR}/{tag}/{c}__rep1.jsonl')[0]
                               < EXPECTED[c]['requests']]
                    print(f'[{tag}] exit {rc}, {dt / 60:.1f} min, incomplete: {missing}', flush=True)
                    run_log['part2'][f'{mode}_min'] = dt / 60
                    ok = rc == 0 and not missing
                    mark(tag, ok, '' if ok else f'exit {rc}, incomplete {missing}')
                except Exception:
                    tb = traceback.format_exc()
                    mark(tag, False, tb)
                    print(tb)
    subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


## Status and hand-back

Lists every planned answer file with its count of answered requests, error lines, the longest prompt in tokens (backbones) and the recorded revision against the pin. A part not run yet shows as MISSING. Then writes `answers_revision3.zip` into `My Drive/Jev/paper4/colab/`.

In [ ]:
import json, os, datetime, zipfile

def scan(path):
    ok = err = 0
    revs, ntok, trunc, errs = set(), 0, 0, {}
    with open(path, encoding='utf-8') as fh:
        for line in fh:
            try:
                j = json.loads(line)
            except json.JSONDecodeError:
                continue
            if j.get('error'):
                err += 1
                e = j['error']
                if 'max_len' in e or 'max_model_len' in e or 'budget' in e:
                    trunc += 1
                errs[e[:80]] = errs.get(e[:80], 0) + 1
                continue
            ok += 1
            revs.add(j.get('revision'))
            for a in j['answers'].values():
                ntok = max(ntok, (a.get('raw') or {}).get('n_prompt_tokens') or 0)
    return ok, err, revs, ntok, trunc, errs

variant = globals().get('COMPARATOR2_VARIANT', 'fp8')
T = COMPARATOR2_TAGS[variant]
planned = [('part 1a', t, OOS) for t in DECISION_TAGS]
if globals().get('RUN_COMPARATOR'):
    planned += [('part 1b', 'comparator-open', OOS)]
if globals().get('RUN_BACKBONE_DESC'):
    planned += [('part 1c', t, c) for t in DESC_TAGS for c in DESC_CONDS]
planned += [('part 2', T[m], c) for m in ('verbal', 'll') for c in PART2_CONDS]

status, problems = {}, []
for part, tag, cond in planned:
    want = EXPECTED[cond]['requests']
    path = f'{ANSWERS_DIR}/{tag}/{cond}__rep1.jsonl'
    key = f'{tag}/{cond}__rep1'
    if not os.path.exists(path):
        status[key] = f'MISSING ({part})'
        problems.append(key)
        continue
    ok, err, revs, ntok, trunc, errs = scan(path)
    pin = REVISIONS[TAG_REPO[tag]]
    rev_ok = revs == {pin}
    note = (f'{ok}/{want} answered, {err} error lines ({trunc} over the length limit), '
            f'revision {"ok" if rev_ok else revs}')
    if ntok:
        note += f', longest prompt {ntok} tokens'
    complete = ok >= want and rev_ok
    status[key] = ('OK ' if complete else 'INCOMPLETE ') + note
    if errs:
        status[key] += f', errors: {errs}'
    if not complete:
        problems.append(key)

for k, v in status.items():
    print(f'{k:62s} {v}')
run_log['end'] = datetime.datetime.utcnow().isoformat() + 'Z'
run_log['status'] = status
run_log['revisions_pinned'] = REVISIONS
with open(f'{ANSWERS_DIR}/run_log_revision3.json', 'w') as fh:
    json.dump(run_log, fh, indent=2)

if problems:
    print(f'\n{len(problems)} file(s) missing or incomplete. A part not run yet shows as MISSING; '
          'otherwise re-run the cell of that model: finished requests are skipped.')
else:
    with open(f'{ANSWERS_DIR}/ALL_DONE_revision3', 'w') as fh:
        fh.write(run_log['end'] + '\n')
    print('\nEvery planned answer file is complete. ALL_DONE_revision3 written.')

# the hand-back archive, next to the folder on Drive: the answer files, the DONE markers and
# run_log_revision3.json; console logs and tracebacks stay on Drive only
archive = f'{DRIVE_ROOT}/answers_revision3.zip'
n_files = 0
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _dirs, files in os.walk(ANSWERS_DIR):
        for fn in sorted(files):
            if fn.endswith('.jsonl') or fn in ('DONE_revision3', 'run_log_revision3.json',
                                               'ALL_DONE_revision3'):
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, ANSWERS_DIR))
                n_files += 1
print(f'hand-back archive: {archive} ({n_files} files)')
